# OSL Dataset Pose Extraction

This notebook extracts pose keypoints from the OSL Dataset videos (MP4 → PKL)

**Folders to process:**
- OSL-Words: 50+ Arabic words/phrases
- OSL-Letters: 28 Arabic letters (أ-ي)
- OSL-Numbers: 0-10 (٠-١٠)
- OSL-Sentences: Sentences

## 1. Setup and Imports

In [1]:
import os
import sys
import cv2
import glob
import pickle
import numpy as np
from tqdm import tqdm
from pathlib import Path

# Add rtmlib to path
rtmlib_path = r'C:\Users\MOBPC\Downloads\FYP\FYPproject\Uni-Sign-main\Uni-Sign-main\demo\rtmlib-main'
if rtmlib_path not in sys.path:
    sys.path.insert(0, rtmlib_path)

from rtmlib import Wholebody, draw_skeleton

print("✓ All imports successful!")

✓ All imports successful!


## 2. Define Paths

In [2]:
# Base paths
OSL_DATASET_ROOT = r'C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\dataset'

# Define source (RGB) and target (Pose) directories for each category
DATASETS = {
    'OSL-Words': {
        'rgb_dir': os.path.join(OSL_DATASET_ROOT, 'OSL-Words', 'rgb_format'),
        'pose_dir': os.path.join(OSL_DATASET_ROOT, 'OSL-Words', 'pose_format'),
    },
    'OSL-Letters': {
        'rgb_dir': os.path.join(OSL_DATASET_ROOT, 'OSL-Letters', 'rgb_format'),
        'pose_dir': os.path.join(OSL_DATASET_ROOT, 'OSL-Letters', 'pose_format'),
    },
    'OSL-Numbers': {
        'rgb_dir': os.path.join(OSL_DATASET_ROOT, 'OSL-Numbers', 'rgb_format'),
        'pose_dir': os.path.join(OSL_DATASET_ROOT, 'OSL-Numbers', 'pose_format'),
    },
    'OSL-Sentences': {
        'rgb_dir': os.path.join(OSL_DATASET_ROOT, 'OSL-Sentences', 'rgb_format'),
        'pose_dir': os.path.join(OSL_DATASET_ROOT, 'OSL-Sentences', 'pose_format'),
    },
}

# Create pose_format directories if they don't exist
for name, paths in DATASETS.items():
    os.makedirs(paths['pose_dir'], exist_ok=True)
    rgb_exists = os.path.exists(paths['rgb_dir'])
    video_count = len(glob.glob(os.path.join(paths['rgb_dir'], '*.mp4'))) if rgb_exists else 0
    print(f"{name}:")
    print(f"  RGB folder exists: {rgb_exists}")
    print(f"  Videos found: {video_count}")
    print(f"  Pose folder: {paths['pose_dir']}")
    print()

OSL-Words:
  RGB folder exists: True
  Videos found: 1235
  Pose folder: C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\dataset\OSL-Words\pose_format

OSL-Letters:
  RGB folder exists: True
  Videos found: 0
  Pose folder: C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\dataset\OSL-Letters\pose_format

OSL-Numbers:
  RGB folder exists: True
  Videos found: 22
  Pose folder: C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\dataset\OSL-Numbers\pose_format

OSL-Sentences:
  RGB folder exists: True
  Videos found: 447
  Pose folder: C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\dataset\OSL-Sentences\pose_format



## 3. Initialize Pose Model

In [3]:
# Initialize the Wholebody pose estimation model (GPU only)
import torch
import onnxruntime as ort

print("Loading Wholebody pose model (GPU-only mode)...")
providers = ort.get_available_providers()
print(f"ONNX providers available: {providers}")

if 'CUDAExecutionProvider' not in providers:
    raise RuntimeError(
        "CUDAExecutionProvider is not available. "
        "Install/check onnxruntime-gpu + CUDA/cuDNN."
    )

# On Windows, explicitly preload CUDA/cuDNN DLLs for ONNX Runtime.
if hasattr(ort, 'preload_dlls'):
    ort.preload_dlls()
    print("ONNX Runtime CUDA DLLs preloaded.")
else:
    print("onnxruntime.preload_dlls() not available; relying on system PATH.")

# Force CUDA only (avoid TensorRT plugin errors on systems without TRT runtime).
GPU_PROVIDER_ORDER = ['CUDAExecutionProvider']

# rtmlib uses this map when creating InferenceSession. Override it so sessions
# are constructed with CUDA explicitly.
import rtmlib.tools.base as rtmlib_base
rtmlib_base.RTMLIB_SETTINGS['onnxruntime']['cuda'] = GPU_PROVIDER_ORDER

wholebody = Wholebody(
    to_openpose=False,      # Use RTMPose format (133 keypoints)
    mode='lightweight',      # Options: 'performance', 'lightweight', 'balanced'
    backend='onnxruntime',   # Use ONNX Runtime
    device='cuda'            # Force GPU path in rtmlib
)

print(f"Wholebody model initialized with provider order: {GPU_PROVIDER_ORDER}")

Loading Wholebody pose model (GPU-only mode)...
ONNX providers available: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Skip loading CUDA and cuDNN DLLs since torch is imported.
ONNX Runtime CUDA DLLs preloaded.
load C:\Users\MOBPC\.cache\rtmlib\hub\checkpoints\yolox_tiny_8xb8-300e_humanart-6f3252f9.onnx with onnxruntime backend
load C:\Users\MOBPC\.cache\rtmlib\hub\checkpoints\rtmw-dw-l-m_simcc-cocktail14_270e-256x192_20231122.onnx with onnxruntime backend
Wholebody model initialized with provider order: ['CUDAExecutionProvider']


## 4. Pose Extraction Functions

In [4]:
# Strict GPU verification (fails if model sessions are not on GPU providers)
import numpy as np
import onnxruntime as ort

print("ONNX Runtime GPU Check:")
providers = ort.get_available_providers()
print(f"  Available providers: {providers}")

gpu_eps = [p for p in ['TensorrtExecutionProvider', 'CUDAExecutionProvider'] if p in providers]
print(f"  GPU providers available: {gpu_eps}")

if not gpu_eps:
    raise RuntimeError("No GPU provider available in ONNX Runtime.")


def _session_providers(model_obj):
    found = []
    # Common session attribute names used by wrappers
    for attr_name in ['session', 'sess', 'ort_session']:
        sess = getattr(model_obj, attr_name, None)
        if sess is not None and hasattr(sess, 'get_providers'):
            found = sess.get_providers()
            break
    return found


def _is_gpu_session(session_provider_list):
    return any(p in ['TensorrtExecutionProvider', 'CUDAExecutionProvider'] for p in session_provider_list)


# Run one warmup inference to trigger actual session execution path
_dummy = np.zeros((256, 256, 3), dtype=np.uint8)
_ = wholebody(_dummy)
print("  Warmup inference OK")

det_providers = _session_providers(getattr(wholebody, 'det_model', None))
pose_providers = _session_providers(getattr(wholebody, 'pose_model', None))

if det_providers:
    print(f"  Detector session providers: {det_providers}")
if pose_providers:
    print(f"  Pose session providers:     {pose_providers}")

# Enforce GPU provider when sessions are introspectable
if pose_providers and not _is_gpu_session(pose_providers):
    raise RuntimeError("Pose model session is not using a GPU execution provider.")
if det_providers and not _is_gpu_session(det_providers):
    raise RuntimeError("Detector session is not using a GPU execution provider.")

if not det_providers and not pose_providers:
    raise RuntimeError(
        "Could not verify provider from rtmlib sessions. "
        "Update rtmlib or inspect model internals before running extraction."
    )

print("GPU verification passed: detector and pose sessions are using GPU providers.")

ONNX Runtime GPU Check:
  Available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
  GPU providers available: ['TensorrtExecutionProvider', 'CUDAExecutionProvider']
  Warmup inference OK
  Detector session providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
  Pose session providers:     ['CUDAExecutionProvider', 'CPUExecutionProvider']
GPU verification passed: detector and pose sessions are using GPU providers.


In [5]:
def process_video(video_path, tgt_dir, wholebody, overwrite=False):
    """Process a single video and save pose data as PKL."""
    output_path = os.path.join(tgt_dir, os.path.basename(video_path).replace(".mp4", ".pkl"))
    
    # Skip if already processed
    if os.path.exists(output_path) and not overwrite:
        return f"Skipped (exists): {os.path.basename(video_path)}"

    data = {"keypoints": [], "scores": []}

    # Open video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return f"Failed to open: {video_path}"

    # Process frames sequentially (GPU-efficient, no CPU threading)
    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame = np.uint8(frame)
        H, W, C = frame.shape
        keypoints, scores = wholebody(frame)
        
        # Normalize keypoints by image dimensions
        data['keypoints'].append(keypoints / np.array([W, H])[None, None])
        data['scores'].append(scores)
        frame_count += 1
    
    cap.release()

    if frame_count == 0:
        return f"No frames in: {video_path}"

    # Save as pickle
    with open(output_path, 'wb') as file:
        pickle.dump(data, file)

    return f"Processed: {os.path.basename(video_path)} ({frame_count} frames)"


def extract_poses_from_folder(rgb_dir, pose_dir, wholebody, overwrite=False):
    """Extract poses from all videos in a folder."""
    video_files = glob.glob(os.path.join(rgb_dir, '*.mp4'))
    
    if len(video_files) == 0:
        print(f"  No MP4 files found in {rgb_dir}")
        return
    
    print(f"  Found {len(video_files)} videos")
    
    for video_path in tqdm(video_files, desc="  Processing"):
        result = process_video(video_path, pose_dir, wholebody, overwrite=overwrite)

print("✓ Functions defined (GPU-optimized, no threading)")

✓ Functions defined (GPU-optimized, no threading)


## 5. Extract Poses for All Datasets

In [ ]:
# Process OSL-Sentences only (Words and Numbers already done)
OVERWRITE_EXISTING = False  # Already-processed files will be skipped

SKIP_DATASETS = ['OSL-Words', 'OSL-Letters', 'OSL-Numbers']

for dataset_name, paths in DATASETS.items():
    if dataset_name in SKIP_DATASETS:
        print(f"⏭ Skipping: {dataset_name}")
        continue

    print(f"\n{'='*60}")
    print(f"Processing: {dataset_name}")
    print(f"{'='*60}")

    if not os.path.exists(paths['rgb_dir']):
        print(f"  ⚠ RGB folder not found: {paths['rgb_dir']}")
        continue

    extract_poses_from_folder(
        rgb_dir=paths['rgb_dir'],
        pose_dir=paths['pose_dir'],
        wholebody=wholebody,
        overwrite=OVERWRITE_EXISTING
    )

print("\n✓ Done!")


Processing: OSL-Words
  Found 1235 videos


  Processing: 100%|██████████| 1235/1235 [3:49:53<00:00, 11.17s/it] 



Processing: OSL-Letters
  No MP4 files found in C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\dataset\OSL-Letters\rgb_format

Processing: OSL-Numbers
  Found 22 videos


  Processing: 100%|██████████| 22/22 [06:24<00:00, 17.49s/it]



Processing: OSL-Sentences
  Found 447 videos


  Processing:   2%|▏         | 11/447 [05:16<3:29:04, 28.77s/it]


KeyboardInterrupt: 

## 6. Verify Results

In [ ]:
# Check extraction results
print("Extraction Summary:")
print("="*60)

for dataset_name, paths in DATASETS.items():
    rgb_count = len(glob.glob(os.path.join(paths['rgb_dir'], '*.mp4')))
    pose_count = len(glob.glob(os.path.join(paths['pose_dir'], '*.pkl')))
    
    status = "✓" if rgb_count == pose_count else "⚠"
    print(f"{status} {dataset_name}:")
    print(f"    RGB videos: {rgb_count}")
    print(f"    PKL files:  {pose_count}")
    
    if rgb_count != pose_count:
        print(f"    Missing: {rgb_count - pose_count} files")
    print()

## 7. Test: Load and Visualize a PKL File

In [ ]:
# Load and inspect a sample PKL file
import matplotlib.pyplot as plt

# Find a sample PKL file
sample_pkl = None
for dataset_name, paths in DATASETS.items():
    pkl_files = glob.glob(os.path.join(paths['pose_dir'], '*.pkl'))
    if pkl_files:
        sample_pkl = pkl_files[0]
        break

if sample_pkl:
    print(f"Loading: {sample_pkl}")
    
    with open(sample_pkl, 'rb') as f:
        data = pickle.load(f)
    
    print(f"\nData structure:")
    print(f"  Keys: {data.keys()}")
    print(f"  Number of frames: {len(data['keypoints'])}")
    print(f"  Keypoints shape per frame: {data['keypoints'][0].shape}")
    print(f"  Scores shape per frame: {data['scores'][0].shape}")
    
    # Visualize first frame keypoints
    frame_idx = 0
    kp = data['keypoints'][frame_idx][0]  # First person in frame
    scores = data['scores'][frame_idx][0]
    
    plt.figure(figsize=(8, 8))
    plt.scatter(kp[:, 0], -kp[:, 1], c=scores, cmap='viridis', s=20)
    plt.colorbar(label='Confidence')
    plt.title(f'Keypoints from {os.path.basename(sample_pkl)} (Frame {frame_idx})')
    plt.xlabel('X (normalized)')
    plt.ylabel('Y (normalized, inverted)')
    plt.axis('equal')
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("No PKL files found yet. Run the extraction first.")

## 8. Process Individual Dataset (Optional)

Use this cell to process a specific dataset only.

In [8]:
# Uncomment and modify to process a specific dataset

DATASET_TO_PROCESS = 'OSL-Sentences'  # Change to: 'OSL-Letters', 'OSL-Numbers', 'OSL-Sentences'

if DATASET_TO_PROCESS in DATASETS:
    paths = DATASETS[DATASET_TO_PROCESS]
    print(f"Processing {DATASET_TO_PROCESS}...")
    extract_poses_from_folder(
        rgb_dir=paths['rgb_dir'],
        pose_dir=paths['pose_dir'],
        wholebody=wholebody,
        overwrite=True  # Set to True to reprocess
    )
    print("Done!")
else:
    print(f"Dataset '{DATASET_TO_PROCESS}' not found.")

Processing OSL-Sentences...
  Found 447 videos


  Processing: 100%|██████████| 447/447 [2:12:47<00:00, 17.82s/it]  

Done!


In [6]:
# ===============================================
# 9. Extract Poses for Augmented Videos (MP4 -> PKL)
# Source : videos_aug/OSL-*/rgb_format/*.mp4
# Target : videos_aug/OSL-*/pose_format/*.pkl
# ===============================================

# Augmented dataset paths in the same style as the original dataset
AUG_DATASET_ROOT = r'C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\videos_aug'

AUG_DATASETS = {
    # 'OSL-Words': {
    #     'rgb_dir': os.path.join(AUG_DATASET_ROOT, 'OSL-Words', 'rgb_format'),
    #     'pose_dir': os.path.join(AUG_DATASET_ROOT, 'OSL-Words', 'pose_format'),
    # },
    # 'OSL-Letters': {
    #     'rgb_dir': os.path.join(AUG_DATASET_ROOT, 'OSL-Letters', 'rgb_format'),
    #     'pose_dir': os.path.join(AUG_DATASET_ROOT, 'OSL-Letters', 'pose_format'),
    # },
    # 'OSL-Numbers': {
    #     'rgb_dir': os.path.join(AUG_DATASET_ROOT, 'OSL-Numbers', 'rgb_format'),
    #     'pose_dir': os.path.join(AUG_DATASET_ROOT, 'OSL-Numbers', 'pose_format'),
    # },
    'OSL-Sentences': {
        'rgb_dir': os.path.join(AUG_DATASET_ROOT, 'OSL-Sentences', 'rgb_format'),
        'pose_dir': os.path.join(AUG_DATASET_ROOT, 'OSL-Sentences', 'pose_format'),
    },
}

# Create output pose folders and show input counts
for name, paths in AUG_DATASETS.items():
    os.makedirs(paths['pose_dir'], exist_ok=True)
    rgb_exists = os.path.exists(paths['rgb_dir'])
    rgb_count = len(glob.glob(os.path.join(paths['rgb_dir'], '*.mp4'))) if rgb_exists else 0
    pose_count = len(glob.glob(os.path.join(paths['pose_dir'], '*.pkl')))
    print(f"{name}:")
    print(f"  RGB folder exists: {rgb_exists}")
    print(f"  Aug videos found: {rgb_count}")
    print(f"  Existing PKL: {pose_count}")
    print(f"  Pose output: {paths['pose_dir']}")
    print()

# Run extraction on augmented videos
OVERWRITE_AUG_EXISTING = False
SKIP_AUG_CATEGORIES = []  # Example: ['OSL-Words']

for category_name, paths in AUG_DATASETS.items():
    if category_name in SKIP_AUG_CATEGORIES:
        print(f"Skipping: {category_name}")
        continue

    print(f"\n{'=' * 60}")
    print(f"Processing augmented: {category_name}")
    print(f"{'=' * 60}")

    if not os.path.exists(paths['rgb_dir']):
        print(f"  RGB folder not found: {paths['rgb_dir']}")
        continue

    extract_poses_from_folder(
        rgb_dir=paths['rgb_dir'],
        pose_dir=paths['pose_dir'],
        wholebody=wholebody,
        overwrite=OVERWRITE_AUG_EXISTING,
    )

print("\nAugmented pose extraction finished!")

# Verify augmented extraction results
print("\nAugmented Extraction Summary:")
print("=" * 60)
for category_name, paths in AUG_DATASETS.items():
    rgb_count = len(glob.glob(os.path.join(paths['rgb_dir'], '*.mp4')))
    pose_count = len(glob.glob(os.path.join(paths['pose_dir'], '*.pkl')))
    status = "OK" if rgb_count == pose_count else "MISSING"
    print(f"{status} {category_name}:")
    print(f"    Aug videos: {rgb_count}")
    print(f"    PKL files:  {pose_count}")
    if rgb_count != pose_count:
        print(f"    Missing: {rgb_count - pose_count}")
    print()

OSL-Sentences:
  RGB folder exists: True
  Aug videos found: 10281
  Existing PKL: 0
  Pose output: C:\Users\MOBPC\Downloads\FYP\FYPproject\OSL_Dataset\videos_aug\OSL-Sentences\pose_format


Processing augmented: OSL-Sentences
  Found 10281 videos


  Processing: 100%|██████████| 10281/10281 [7:31:22<00:00,  2.63s/it]  


Augmented pose extraction finished!

Augmented Extraction Summary:
OK OSL-Sentences:
    Aug videos: 10281
    PKL files:  10281

